In [1]:
import pyhidra

# FYI: this "initializes the application" (per Ghidra docs)
launcher = pyhidra.start()

In [2]:
#!echo $PATH | tr ':' '\n'
import typing
if typing.TYPE_CHECKING:
    import ghidra
    from ghidra.ghidra_builtins import *

import ghidra
from ghidra.base.project import GhidraProject

# ghidra://localhost/binutils/run1.gcc-O0.binutils-2_36
# repo.fileExists('/run1.gcc-O0.binutils-2_36', '0.gdb.debug')

host = 'localhost'
port = 13100
repoName = 'astera'
folderPath = '/run1.gcc.astera'
binaryName = '0.fighter.debug'


In [3]:
from ghidralib import OpenSharedGhidraProject

# TODO: NORMAL WAY:
# with OpenSharedGhidraProject(host, repoName, port) as proj:

# TEMP: do it this way so I can keep state in separate cells...
sgp = OpenSharedGhidraProject(host, repoName, port)
proj = sgp.__enter__()

prog = proj.openProgram(folderPath, binaryName, True)
fm = prog.getFunctionManager()
nonthunks = (x for x in fm.getFunctions(True) if not x.isThunk())

test_func = [x for x in nonthunks if x.name == 'main'][0]
print(test_func.name)

main


In [4]:
from ghidra.app.decompiler import DecompInterface, DecompileOptions

from ghidralib import get_decompiler_interface

ifc = get_decompiler_interface(prog)

test_func.signature
DECOMPILE_TIMEOUT_SEC = 180
res = ifc.decompileFunction(test_func, DECOMPILE_TIMEOUT_SEC, None)

In [5]:

def remove_comments_and_blank_lines(ghidra_c:str) -> str:
    '''
    Remove comment lines and blank lines from Ghidra C code to help our AST diff match
    '''
    return '\n'.join([l for l in ghidra_c.split('\n') if l.strip() and l.strip()[:2] != '/*'])

with open(f'{test_func.name}.ghidra.c', 'w') as f:
    f.write(remove_comments_and_blank_lines(res.getDecompiledFunction().getC()))

In [6]:
# TODO: write my version...
with open(f'{test_func.name}.ast.c', 'w') as f:
    f.write('\n/* WARNING: Removing unreachable block (ram,0x0014e339) */')

# Remove unwanted nodes from OLD AST JSON

In [7]:
# import json
# with open('OLD-AST-main.json', 'r') as f:
#     data = json.load(f)

# new_dict = {'kind': data['kind'], 'inner': []}

# skip_kinds = ['EnumDecl', 'RecordDecl']
# for x in data['inner']:
#     if x['kind'] in skip_kinds:
#         continue
#     new_dict['inner'].append(x)
    # print(x['kind'])

# with open('OLD-AST-2-main.json', 'w') as f:
#     json.dump(new_dict, f, indent=2)

In [40]:
from ghidra.program.model.pcode import PcodeBlockBasic
from ghidra.app.decompiler import *

bb:PcodeBlockBasic = res.getHighFunction().basicBlocks[0]
token_group = res.getCCodeMarkup()

child_nodes = [token_group.Child(i) for i in range(token_group.numChildren())]
child_nodes
#if not isinstance(token_group.Child(i), ClangSyntaxToken)]

# TODO: pick up here walking the AST (via child_nodes...)
# TODO - go ahead and define TranslationUnitDecl, we need one to start!
# TODO - go ahead and define AstBuilder - this will hold our state...

[,
 ,
 /* ,
 ,
  ,
 Removing,
  ,
 unreachable,
  ,
 block,
  ,
 (ram,0x0014e339),
 ,
  */,
 ,
 /* ,
 ,
  ,
 Removing,
  ,
 unreachable,
  ,
 block,
  ,
 (ram,0x0014e367),
 ,
  */,
 ,
 /* ,
 ,
  ,
 Unknown,
  ,
 calling,
  ,
 convention,
 ,
  */,
 ,
 ,
 ,
 int main(void),
 ,
 ,
 ,
 {,
 ,
 long lVar1,
 ,
 ;,
 ,
 r_framebuffer fbo,
 ,
 ;,
 ,
 r_framebuffer fbo_00,
 ,
 ;,
 ,
 r_window_params params_00,
 ,
 ;,
 ,
 undefined auVar2 [32],
 ,
 ;,
 ,
 a_ctx_info ctx_info_00,
 ,
 ;,
 ,
 undefined auVar3 [24],
 ,
 ;,
 ,
 uint8_t uVar4,
 ,
 ;,
 ,
 int iVar5,
 ,
 ;,
 ,
 time_t tVar6,
 ,
 ;,
 ,
 long in_FS_OFFSET,
 ,
 ;,
 ,
 time_s extraout_XMM0_Qa,
 ,
 ;,
 ,
 time_s extraout_XMM0_Qa_00,
 ,
 ;,
 ,
 time_s tVar7,
 ,
 ;,
 ,
 time_s in_XMM1_Qa,
 ,
 ;,
 ,
 asset_t *local_a8,
 ,
 ;,
 ,
 undefined8 local_a0,
 ,
 ;,
 ,
 undefined8 local_98,
 ,
 ;,
 ,
 undefined8 local_90,
 ,
 ;,
 ,
 time_s delta,
 ,
 ;,
 ,
 time_s render_delta,
 ,
 ;,
 ,
 a_ctx_info ctx_info,
 ,
 ;,
 ,
 r_window_params params,
 ,
 ;,
 ,
 

In [41]:
sgp.__exit__(None, None, None)